# Classification of Wikipedia Articles
Wikipedia is an encyclopedia that covers a large amount of diverse topics. All articles are created, corrected and updated by individuals. The goal is to correctly document as many topics as possible by collecting the knowledge of a large number of people. However, some articles stand out due to their completeness, scope and presentation, and for this they are marked with the distinction of the Excellent Article. 

As part of the Natural Language Processing lecture, a classification of Wikipedia articles is to be carried out as a sub-task of an assignment with the goal of being able to identify excellent articles. This notebook contains the code to accomplish this goal and is structured as follows:

1. [Imports](#1-imports)
2. [Automated Data Check](#2-check-data-availability)
3. [Load and Process the Data](#3-data-processing)
4. [Save Preprocessed Data](#4-save-preprocessed-data)

## 1. Imports
Import the requiered libraties into the notebook.
If some libraries are not installed, you can use the `requierements.txt` and run
```
$ pip install -r requirements.txt
```
in the terminal.

In [1]:
# Import buildin libraries
import os
import subprocess
import shutil

# Import wikipedia dump specific libraries
import mwxml
import mwparserfromhell

# Import data science libraries
import re
import pandas as pd
import numpy as np

# Import resampling library
from sklearn.utils import resample

# Import progress bar library
from tqdm.notebook import tqdm

## 2. Check Data Availability
In order to get the training data, a backup of the current wikipedia encyclopedie is needed. <br>
These dumps can be downloaded in every language by changing the url to https://dumps.wikimedia.org/[`insert_language (e.g. de, en)`]wiki/latest/. <br>
To make this example easy to run and have an equal data foundation, the download was automated with this cell. If the correct file already exists in the data folder, the download will be skipped. <br>
The file will be donwloaded as an `.bz2`-Archive and must be extracted before use. <br>
<b>Note: Due to the size of the file, the download may take a longer time depending on your internet connection.</b>

In [3]:
if(not(os.path.exists("../../Data"))):
   os.makedirs('../../Data')

if(not(os.path.exists("../../Data/dewiki-latest-pages-articles-multistream.xml"))):
    print("Articles-File not found. Download started... (this might take a while):")
    subprocess.call(['sh', '../../bin/download-and-unzip-data.sh'])
print("Articles-File available at: ../Data/dewiki-latest-pages-articles-multistream.xml")

Articles-File available at: ../Data/dewiki-latest-pages-articles-multistream.xml


## 3. Data Processing
A big challenge is the effecient processing of the wikipedia dump files. Due to its size (ca. 26,8 GB), it is not possible to load the whole file into the memory. There are different approaches to deal with this problem but we chose to use the `mwxml`-library, which creates an generator-object that returns a single Wikipedia article at a time. In order to use natural language processing for the classification of the individual articles, the respecitve texts must be extracted. However, this poses another challenge due to the HTML-formatting.

In [4]:
def dump_to_dataset(path_to_dump:str, n_samples:int=-1, balance_ratio:float=-1.0, random_state:int=456) -> pd.DataFrame:
    """
    Creates a dataset out of a wikipedia dump that contains the text and label of each article.

    If the argument 'n_samples' is passed, only the specified number of samples will be processed.

    If the argument 'balance_ratio' is passed, the dataset will be resampled in order to achieve the desired ratio.

    Parameters
    ----------
    path_to_dump : str. Relative file path to the unzipped wikipedia dump
    n_samples: int, optional.  Samples to be processed, if not set the whole dataset will be iterated
    balance_ratio: float, optional. Desired resampling ratio of majority and minority class, if not set no resampling will be performed
    random_state: int, optional. Random state for resampling, if not set default random state will be used

    Returns
    -------
    df: pd.DataFrame. Pandas DataFrame that contains the processed wikipedia articles with labels
    """
    data_index = [] # Empty array for the article ids
    label_index = [] # Empty array for the labels
    dump = mwxml.Dump.from_file(open(path_to_dump)) # Load Wikipedia dump and create generator object
    i = 0 # Set iteration variable to zero
    print("Step 1: Indexing Wikipedia Dataset")
    pbar = tqdm(total= (n_samples if (n_samples>1) else 5425758)) # Create statusbar
    for page in dump: # Iterate over pages in dump
        for revisions in page: # Iterate over revisions of page
            try:
                if((revisions.page.namespace == 0) & (not revisions.page.redirect) & ("Liste" not in revisions.page.title)):
                    if(re.search(r"{{Exzellent[|](\d*).(\D*)(\d*)[|](\d*)}}", revisions.text)): # If article is marked as excelennt
                        label_index.append(1) # Add 1 (positive) as label to array
                    else:
                        label_index.append(0) # Add 0 (negative) as label to array
                    data_index.append(revisions.page.id) # Add page id to array
                    i += 1 # Increment iteration variable
                pbar.update(1) # Updata status bar
            except Exception as e:
                print(e)
        if(i>=(n_samples if (n_samples>1) else 5425758)): # If n_samples or end of dump is reached
            break # End iterations
    print(len(data_index))
    pbar.close() # Stop progress bar
    print("Step 2: Resample Collected Data")
    temp_df = pd.DataFrame({'text_id': data_index, 'label': label_index}, columns=['text_id', 'label']) #  Create temporal dataframe
    minority_class = temp_df[temp_df['label'] == 1]
    downsampled_majority = resample(
        temp_df[temp_df['label'] == 0], # Get majority class samples
        replace=False, # Undersampling
        n_samples=int(balance_ratio * len(minority_class)), # Set resampling ratio
        random_state=random_state # Add random state
    )
    downsampled_df = pd.concat([downsampled_majority, minority_class]).sample(frac=1) # Combine downsampled majority class with minority class
    valid_list = downsampled_df["text_id"].values # Create list with valid article ids
    print("Step 3: Create Balanced Dataset")
    os.makedirs('../../Data/tmp') # Create processed data folder
    text = np.memmap('../../Data/tmp/article_text.dat', dtype='object', mode='w+', shape=(len(valid_list), 1)) # Create memmap for texts
    label = np.memmap('../../Data/tmp/label.dat', dtype=np.int8, mode='w+', shape=(len(valid_list), 1)) # Create memmap for labels
    dump = mwxml.Dump.from_file(open(path_to_dump)) # Load Wikipedia dump and create generator object
    i = 0 # Reset iteration variable to zero
    pbar2 = tqdm(total=len(valid_list)) # Create statusbar
    for page in dump: # Iterate over pages in dump
        for revisions in page: # Iterate over revisions in page
            try:
                if("Liste" not in revisions.page.title): # Exclude non-article revisions
                    if revisions.page.id in valid_list: # If article is in resampled scope
                        temp_text = mwparserfromhell.parse(revisions.text) # Parse text
                        if any((re.match(r"{{Exzellent[|](\d*).(\D*)(\d*)[|](\d*)}}", str(template)) for template in temp_text.filter_templates())): # If article is markes as excellent
                            label[i] = 1 # Add 1 (positive) as label to memmap
                        else:
                            label[i] = 0 # Add 0 (negative) as label to memmap
                        text[i] = re.sub(r"Exzellent (\d*).(\D*).(\d*)", "", (" ".join((" ".join(list(map(str, temp_text.filter_text())))).split()))) # Add cleaned text to memmap
                        i += 1 # Increment iteration variable
                        pbar2.update(1) # Update status bar
            except Exception as e:
                print(e)
        if(i>=len(valid_list)): # If valid list is completed
            break # Break iteration
    pbar2.close() # Close status bar
    df = pd.DataFrame({'text': np.array(text).ravel(), 'label': list(np.array(label))}, columns=['text', 'label']) # Combine memmap arrays to dataframe
    shutil.rmtree('../../Data/tmp')
    return df # return dataframe

In [5]:
path_to_dump = "../../Data/dewiki-latest-pages-articles-multistream.xml"
dataframe = dump_to_dataset(path_to_dump=path_to_dump, balance_ratio=1.5)

Step 1: Indexing Wikipedia Dataset


  0%|          | 0/5425758 [00:00<?, ?it/s]

2723060
Step 2: Resample Collected Data
Step 3: Create Balanced Dataset


  0%|          | 0/6987 [00:00<?, ?it/s]

## 4. Save Preprocessed Data

In [6]:
dataframe.to_pickle("../../Data/processed_dataset.pkl")